In [28]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import scale
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from scipy import stats
from IPython.display import display, HTML
import math
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier
import joblib



In [29]:
df = pd.read_csv("Depression Dataset.csv")

In [30]:
def label_encoding(df):
    
    feature_names = df.columns.tolist()
    #feature_names.remove("AGERNG")
    feature_names.remove("DEPRESSED")
    label_encoder = LabelEncoder()
    encoded_df = df

    for feature in feature_names:
        df[feature] = label_encoder.fit_transform(df[feature])

    encoded_df = df
    return encoded_df

In [31]:
encoded_df = label_encoding(df)

In [32]:
def print_confusion_matrix(y_test, y_pred):
  # Assuming y_test and y_pred are your true labels and predicted labels respectively
  conf_matrix = confusion_matrix(y_test, y_pred)

  # Extracting values from confusion matrix
  TN, FP, FN, TP = conf_matrix.ravel()

  print("True Positives:", TP)
  print("True Negatives:", TN)
  print("False Positives:", FP)
  print("False Negatives:", FN)

  return

In [33]:
def calculate_evaluation_metrics(y_pred, y_test):

  # Calculate accuracy
  accuracy = accuracy_score(y_test, y_pred)
  accuracy *= 100
  print("Accuracy: ", accuracy, "%")

  # Calculate precision
  precision = precision_score(y_test, y_pred)
  precision *= 100
  print("Precision:", precision, "%")

  # Calculate recall (sensitivity)
  recall = recall_score(y_test, y_pred)
  recall *= 100
  print("Recall (Sensitivity):", recall, "%")

  # Calculate specificity
  conf_matrix = confusion_matrix(y_test, y_pred)
  tn, fp, fn, tp = conf_matrix.ravel()
  specificity = tn / (tn + fp)
  specificity *= 100
  print("Specificity:", specificity, "%")

  # Calculate F1-score
  f1 = f1_score(y_test, y_pred)
  f1 *= 100
  print("F1-score:", f1, "%")

  return accuracy, precision, recall, specificity, f1

In [34]:
def draw_ROC_curves(X_test, y_test, model, title_extra):

  plt.figure()

  y_pred = model.predict_proba(X_test)[:, 1]
  fpr, tpr, thresholds = roc_curve(y_test, y_pred)
  roc_auc = auc(fpr, tpr)
  plt.plot(fpr, tpr, lw=2, label='%s (AUC = %0.2f)' % ('XGB_Model', roc_auc))


  plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
  plt.xlim([0.0, 1.0])
  plt.ylim([0.0, 1.05])
  plt.xlabel('False Positive Rate')
  plt.ylabel('True Positive Rate')
  plt.title('Receiver Operating Characteristic (ROC)' + title_extra)
  plt.legend(loc="lower right", bbox_to_anchor=(1.0, 0.0), fontsize='x-small')
  plt.show()

  return

In [ ]:
def split(df):

    #X = df.drop('DEPRESSED', axis=1)
    #y = df['DEPRESSED']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    return X_train, X_test, y_train, y_test

In [36]:
encoded_df = label_encoding(df)
X_train, X_test, y_train, y_test = split(encoded_df)

In [37]:
def Select_KBest_Feature_Selection(X_train, y_train, num_features=15):

  selector = SelectKBest(score_func=f_classif, k=num_features)
  X_selected = selector.fit_transform(X_train, y_train)

  #selected_feature_names = X.columns[selector.get_support(indices=True)]
  #print(selected_feature_names)
  selected_feature_names = []
  selected_feature_indices = np.array(selector.get_support(indices=True))
  for index in selected_feature_indices:
    selected_feature_names.append(X_train.columns[index])

  print(selected_feature_names)

  df_KBest= pd.DataFrame(X_selected, columns=selected_feature_names)

  return df_KBest, selected_feature_names

df_KBest, selected_feature_names = Select_KBest_Feature_Selection(X_train, y_train, 15)


['ENVSAT', 'POSSAT', 'FINSTR', 'DEBT', 'EATDIS', 'INSOM', 'ANXI', 'DEPRI', 'ABUSED', 'CHEAT', 'THREAT', 'SUICIDE', 'INFER', 'CONFLICT', 'LOST']


In [45]:
X = encoded_df[selected_feature_names]
y = encoded_df['DEPRESSED']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [47]:
def class_balancing(df):

    smote = SMOTE(random_state=42)

    X_train, X_test, y_train, y_test = split(df)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    return X_train_resampled, y_train_resampled

X_train_resampled, y_train_resampled = class_balancing(encoded_df)

In [48]:
def XGB_Classification(X_train, X_test, y_train, y_test):

  xgb_model = xgb.XGBClassifier()

  start_time = time.time()

  xgb_model.fit(X_train, y_train)

  end_time = time.time()

  duration = end_time - start_time

  y_pred = xgb_model.predict(X_test)

  #accuracy = accuracy_score(y_test, y_pred)
  #print("Accuracy:", accuracy*100, "%")

  accuracy, precision, recall, specificity, f1 = calculate_evaluation_metrics(y_pred, y_test)

  # Additional metrics
  print(classification_report(y_test, y_pred))

  print_confusion_matrix(y_test, y_pred)

  return accuracy, xgb_model, duration

In [49]:
selector = SelectKBest( score_func=f_classif, k=15 )
smote = SMOTE( random_state=42 )
classifier = XGBClassifier( random_state=42, eval_metric="logloss", n_estimators=200, 
max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8 )

In [50]:
pipeline = Pipeline(
    steps = [
        ("feature_selection", selector),
        ("class_balancing", smote),
        ("xgb_classifier", classifier)
        ]
)

In [51]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict( X_test )

In [52]:
#y_pred = xgb_model.predict(X_test)
accuracy, precision, recall, specificity, f1 = calculate_evaluation_metrics(y_pred, y_test)
print(classification_report(y_test, y_pred))
print_confusion_matrix(y_test, y_pred)

Accuracy:  88.42975206611571 %
Precision: 97.01492537313433 %
Recall (Sensitivity): 84.4155844155844 %
Specificity: 95.45454545454545 %
F1-score: 90.27777777777779 %
              precision    recall  f1-score   support

           0       0.78      0.95      0.86        44
           1       0.97      0.84      0.90        77

    accuracy                           0.88       121
   macro avg       0.87      0.90      0.88       121
weighted avg       0.90      0.88      0.89       121

True Positives: 65
True Negatives: 42
False Positives: 2
False Negatives: 12


In [55]:
joblib.dump( pipeline, "xgb_depression_pipeline.joblib" )
print( "\nPipeline saved to " "xgb_depression_pipeline.joblib" )


Pipeline saved to xgb_depression_pipeline.joblib
